# 🛡️ Aegis — Colab Model Training Notebook

**Multi-Agent RL for Autonomous Kubernetes Cluster Self-Healing**

This notebook executes the two training stages for Aegis:
1. **Phase 3** — GNN Encoder (self-supervised pretraining + linear probe gate)
2. **Phase 4** — MAPPO Policy (400 PPO updates on GPU + baseline comparison)

Runtime: **~1–2 hours on a free Google Colab T4 GPU**

---

## Cell 1 — Environment Setup (~2 min)
Mount Google Drive, clone or link the repository, and install GPU PyTorch with all dependencies.

In [ ]:
# ============================================================
# 1. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# 2. Repository Setup
# ============================================================
import os

REPO_URL = "https://github.com/Shakti8125/Aegis.git"
REPO_DIR = "/content/Aegis"

if not os.path.exists(REPO_DIR):
    if os.path.exists("/content/drive/MyDrive/Aegis"):
        print("Linking existing Aegis directory from Drive...")
        !cp -r /content/drive/MyDrive/Aegis /content/Aegis
    else:
        print(f"Cloning from {REPO_URL}...")
        !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ============================================================
# 3. Install Dependencies
# ============================================================
!pip install -q numpy==2.5.1 gymnasium==1.3.0 pettingzoo==1.26.1 pytest==9.1.1
!pip install -q torch==2.9.1+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install -q torch-geometric==2.8.0.post1
!pip install -q pyg-lib torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.9.1+cu124.html

# ============================================================
# 4. Verify GPU & Test Suite
# ============================================================
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not enabled. Go to Runtime -> Change runtime type -> T4 GPU")

!python -m pytest tests/simulator/ tests/encoder/ tests/marl/ -q --tb=short 2>&1 | tail -5

---
## Cell 2 — Phase 3: GNN Encoder Pretraining + Probe Gate (~15–25 min)

Runs the Phase 3 validation gate (`encoder.probe`):
1. Collects simulator rollouts across cluster sizes (8, 12, 16 services)
2. Self-supervised pretraining (masked feature reconstruction + link prediction)
3. Freezes encoder weights, fits linear probe classifier on health status
4. Evaluates accuracy on training sizes AND held-out cluster sizes (6, 20, 28 services)

In [ ]:
%%time
# ============================================================
# Phase 3: Execute GNN Probe Gate
# ============================================================
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "encoder.probe"],
    cwd="/content/Aegis",
    capture_output=False,
    text=True,
)

print(f"\n{'='*60}")
if result.returncode == 0:
    print("✅ PHASE 3 GATE: PASSED")
    print("   Encoder embeddings are linearly separable on health categories")
    print("   across both training and held-out cluster sizes. Proceeding to Phase 4.")
else:
    print("❌ PHASE 3 GATE: FAILED")
    print("   Linear probe did not meet precision thresholds.")
print(f"{'='*60}")

---
## Cell 3 — Phase 4: MAPPO Training & Baseline Evaluation (~45–90 min)

Trains hand-rolled MAPPO policy on CUDA:
- 400 updates (409,600 env steps / 4.9M agent steps)
- Scenario: `mixed` (all 5 cluster fault types)
- Automated baseline hyperparameter tuning
- End-of-run comparison against rule-based controller and no-op baseline

In [ ]:
%%time
# ============================================================
# Phase 4: MAPPO GPU Training Run
# ============================================================
!python -m marl.train \
    --updates 400 \
    --rollout-steps 128 \
    --envs 8 \
    --device cuda \
    --train-scenario mixed \
    --eval-every 50 \
    --checkpoint-every 50 \
    --tune-baseline \
    --run-id colab-gpu-v1 \
    --seed 20240401

---
## Cell 4 — Save Checkpoints to Google Drive

Copies trained checkpoints (`final.pt`, `comparison.json`, `metrics.jsonl`) to Google Drive.

In [ ]:
# ============================================================
# Sync Checkpoints to Google Drive
# ============================================================
import shutil, os, json

RUN_ID = "colab-gpu-v1"
src = f"/content/Aegis/marl/checkpoints/{RUN_ID}"
dst = f"/content/drive/MyDrive/Aegis/marl/checkpoints/{RUN_ID}"

os.makedirs(dst, exist_ok=True)
print(f"Saving trained model weights & metrics to Google Drive...\n")

for f in sorted(os.listdir(src)):
    fpath = os.path.join(src, f)
    sz = os.path.getsize(fpath)
    shutil.copy2(fpath, dst)
    print(f"  ✓ {f:30s} ({sz:,} bytes)")

# Print Final Verdict
comp_file = os.path.join(src, "comparison.json")
if os.path.exists(comp_file):
    comp = json.loads(open(comp_file).read())
    won = comp.get('scenarios_won_on_both', 0)
    total = comp.get('scenarios_total', 0)
    print(f"\n{'='*60}")
    print(f"MAPPO vs Baseline Results:")
    print(f"  Scenarios won on both TTR and SLA: {won}/{total}")
    print(f"  Status: {'PASSED ✅' if won > 1 else 'NEEDS MORE UPDATES ❌'}")
    print(f"{'='*60}")

print(f"\nUpload/Sync Complete! Path: {dst}")